In [38]:
import yaml, torch, sys
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torch.nn.functional as F
import random as random
import re
import torch, json, random, os
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, TensorDataset
import sys

from types import SimpleNamespace
#add more packages here
sys.path.insert(0, '../../')
from utils.train_model import run_experiment, set_seed, get_device

In [36]:
import importlib
import utils.train_model
importlib.reload(utils.train_model)
from utils.train_model import run_experiment, set_seed, get_device

In [52]:

sys.path.insert(0, '../../')
from utils.train_model import set_seed, get_device

with open('../../configs/defaults.yaml') as f:
    config = yaml.safe_load(f)

with open('../../configs/EX_01.yaml') as f:
    config.update(yaml.safe_load(f))
    
    

def to_namespace(d):
    if isinstance(d, dict):
        return SimpleNamespace(**{k: to_namespace(v) for k, v in d.items()})
    return d

config = to_namespace(config)

set_seed(config.seed)
device = get_device()

RAW_DIR  = Path(config.raw_dir)
GRAD_DIR = Path(config.grad_dir)


data_dir = Path("./EX_01/Raw/")



[device] using cpu


# Context.

This is the first experiment of the first epic, whose purpose is to establish a baseline. The purpose of EX_01 is to establish the most minimum baseline version of this paper. It is expected that performance will be very poor, but we want to validate and be sure that what we wish to do is doable, before we start optimizing.

First, what we are learning mathematically is a optimal mask over a glyph, which both maximally reduces the probabilty of recognition before the application of a blur, and maximizes the probability of recognition after the application of this blur. For the purposes of this paper, we restrict the possible blurs to blurs that may be modeled by a human hand holding a camera of relatively standard quality - the vaugeness of this description allows for a sufficiently robust set of possible blurs. 

# Hypothesis

Thus, our formal hypothesis is as follows:

A 3-layer "standard" CNN can achieve a higher combined score than a manual approach. Here, "combined score" can be represented by the following mathematical function:

$\Delta-VIS(b, i) = -\alpha * p(x) + \beta * p(y)$

Where $x$ is confidence prior to the blur, and $y$ is posterior, and a $(b, i)$ pair is a background and hidden image pair. 

# Implementation 

$p(x)$ is measured by a simple classifer trained on the MNIST dataset. It consists of two convolutional layers and three linear layers, and achieves an accuracy of 99% after 10 epochs. All layer weights and frozen and will be used as is throughout all experiments unless mentioned otherwise. $p$ is the confidence of the model that the given image is depicting the correct number. 

The "standard CNN" will be a classifier with a single convolutional "block", consisting of a 

# Data

Each background is a sythentically created gradient between two random colors, in a 28x28p image. Each image will be a randomly selected item of the MNIST dataset.

For right now, we're just going to plop the mnist image (in the code called "i" or "image") over the gradient background (in the code called "b" or "background"), forming a 4-channel image (called "c" or "comp" in the code). Future experiments should expand on this by learning other combinations. 

# Notes

It is expected that some $(b, i)$ pairs will work better than others, and we hope we will have sufficient time for the project to also build a classifer to determine this, or else incorporate it in our given model. A useful extension, which is unlikely to be completed in our work, is to automatically build the best possible background. 

# Code

## MNIST Classifer
Here we build the function m_classify, and ensure it works correctly:

In [40]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        x = F.avg_pool2d(F.relu(self.conv1(x)), 2)
        x = F.avg_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))
    
m_model = LeNet5().to(device)
m_model.load_state_dict(torch.load('Model/MNIST/weights/lenet5_mnist.pt', map_location=device))
m_model.eval()

def m_classify(image, correct):
    with torch.no_grad():
            logits = m_model(image)
            preds = torch.softmax(logits, dim=1)
            return preds[0, correct].item()


/scratch/slurm-8299564/ipykernel_2989596/3653177641.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m_model.load_state_dict(torch.load('Model/MNIST/weights/lenet5_mnist

In [6]:
mnist = datasets.MNIST('../../data/', train=False, download=False,
                       transform=transforms.ToTensor())

idx = random.randint(0,10)
digit, label = mnist[idx]
digit = digit.unsqueeze(0).to(device)  # (1, 1, 28, 28)

conf = m_classify(digit, label)
print(f"true: {label}, conf: {conf:.4f}")

true: 2, conf: 0.9850


Seems decent. Let's build the "manual" method. I personally don't know hardly anything about what a good mask would look like, so let's just use a random one. Surely almost anything intentional will be better than this. If we can't beat this, we've got serious problems on our hands.

In [41]:
# pipeline
# will load the mappings into a dataset, create the pipeline function, and then report and

#dataset



class MNIST_GRADIENT_DATASET(Dataset):
    def __init__(self, batch_size, transform=None):
        self.pairings = pd.read_csv(Path(config.csv_path))
        self.gradients_dir = Path(config.grad_dir)
        self.transform = transform if transform is not None else transforms.ToTensor()
        self.mnist     = datasets.MNIST('../../data/', train=True, download=False,
                                        transform=transforms.ToTensor())

    def __len__(self):
        return len(self.pairings)

    def __getitem__(self, idx):
        # ann = self.pairings[pd.where(self.pairings["mnist_idx" == idx])
        ann = self.pairings.iloc[idx]
        b_img_path =  ann["gradient_path"]
        b = Image.open(b_img_path).convert("RGB")
        i, _ = self.mnist[int(ann["mnist_idx"])]

        # Apply image transformation if specified.
        if self.transform:
            b = self.transform(b)

        gold = ann["label"]
        
        match = re.search(r'grad_(\d+)', b_img_path)
        grad_idx = int(match.group(1))
        i_idx   = int(ann["mnist_idx"])
        gold = int(ann["label"])
        # return {"b": b, "i": i,  "gold": gold, "b_idx" : grad_idx, "i_idx" : i_idx}
        return b, i, gold, grad_idx, i_idx

ex01_trainset = MNIST_GRADIENT_DATASET(batch_size = config.batch_size)
ex01_trainloader = DataLoader(ex01_trainset, batch_size=config.batch_size, shuffle=True)
print("Num training pairs: ", len(ex01_trainset))


Num training pairs:  600000


In [56]:

# ── DUMMY MODEL ───────────────────────────────────────────────────────────────
# no layers — just applies a box blur and passes through
#overall pipeline is the model takes in an b and an i, learn a mask for i, loss function actually handles everything at this point.

    
class RandomMask(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = config.variants.a.dropout_rate
    
    def forward(self, x):
        # # x is a 4 channel image, only masking the mnist digit
        # bg    = x[:, :3, :, :]   # (B, 3, 28, 28)
        # digit = x[:, 3:, :, :]   # (B, 1, 28, 28)
        
        mask          = (torch.rand_like(digit) > self.p).float()
        # masked_digit  = digit * mask
        composite = x * mask
        
        
        # composite = bg + masked_digit.expand_as(bg)
        return composite

def box_blur(x, kernel_size=9):
    channels = x.shape[1]
    kernel = torch.ones(channels, 1, kernel_size, kernel_size, 
                        device=x.device) / (kernel_size ** 2)
    return F.conv2d(x, kernel, padding=kernel_size//2, groups=channels)

# ── DUMMY LOSS ────────────────────────────────────────────────────────────────
def ex_01a_loss_fn(batch, model, mode="train", **kwargs):
    b, i, label, b_idx, i_idx = batch

    # comp = torch.cat([b, i], dim=1) #compositing by layering the i over the b
    comp = (b + i.expand_as(b)).clamp(0, 1) #[3, 28, 28]
    masked = model(comp)
    conf_pre  = m_classify(masked, label)
    
    blurred = box_blur(masked)
    conf_post = m_classify(blurred, label)
    delta_vis = (config.variants.a.alpha * conf_pre) + (config.variants.a.beta * conf_post)
    loss = (-delta_vis).mean() if delta_vis.dim() > 0 else -delta_vis
    if mode == "val":
        return {"val_loss": loss.item(), "conf_pre": conf_pre, "conf_post": conf_post, "delta_vis": delta_vis}
    return loss


# ── RUN ───────────────────────────────────────────────────────────────────────
ex01a_info = {
    "exp_id":      "EX_01a",
    "epochs":      3,
    "log_to":      "both",   # keep wandb off until pipeline is verified
    "weights_dir": "Model/A",
    "log_dir":     "logs/A",
    "seed":        42,
}

r_model = RandomMask()
optimizer = torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=1e-3)

run_experiment(
    model=r_model,
    optimizer=optimizer,
    loss_fn=ex_01a_loss_fn,
    train_loader=ex01_trainloader,
    val_loader=ex01_trainloader,
    config=ex01a_info,
    log_to=ex01a_info["log_to"],
)

[device] using cpu


epoch,▁▅█
train_loss,█▁▇
val_loss,▁▂█
epoch,3
train_loss,0.13256
val_loss,0.13311


RandomMask()

The above code section has a glaring issue in that we trained the LeNet on a single channel image, but now we are asking it to classify a 3-channel image. We'll retrain a new one here, which is probalby safer for the sake of reproduction to begin with. 

In [10]:
class LeNet4(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1   = nn.Linear(16*5*5, 120)
        self.fc2   = nn.Linear(120, 84)
        self.fc3   = nn.Linear(84, 10)

    def forward(self, x):
        x = F.avg_pool2d(F.relu(self.conv1(x)), 2)
        x = F.avg_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))



class ex01b(Dataset):
    def __init__(self, batch_size, transform=None):
        self.pairings = pd.read_csv(Path(config.csv_path))
        self.gradients_dir = Path(config.grad_dir)
        self.transform = transform if transform is not None else transforms.ToTensor()
        self.mnist     = datasets.MNIST('../../data/', train=True, download=False,
                                        transform=transforms.ToTensor())

    def __len__(self):
        return len(self.pairings)

    def __getitem__(self, idx):
        # ann = self.pairings[pd.where(self.pairings["mnist_idx" == idx])
        ann = self.pairings.iloc[idx]
        b_img_path =  ann["gradient_path"]
        b = self.transform(Image.open(b_img_path).convert("RGB")) 
        i, lab = self.mnist[int(ann["mnist_idx"])]
        comp = b + i.expand_as(b) #[3, 28, 28]
        return comp.clamp(0, 1), lab

ex01b_trainset = ex01b(batch_size = config.batch_size)
ex01b_trainloader = DataLoader(ex01b_trainset, batch_size=config.batch_size, shuffle=True)
ex01b_valloader = DataLoader(ex01b_trainset, batch_size=config.batch_size, shuffle=True)


In [57]:
config.batch_size = 265 #change this after this run, just to test

In [ ]:
def ex01b_loss_fn(batch, model, mode="train", **kwargs):
    imgs, labels = batch
    logits = model(imgs)
    loss = nn.CrossEntropyLoss()(logits, labels)

    if mode == "val":
        acc = (logits.argmax(1) == labels).float().mean().item()
        return {"val_loss": loss.item(), "val_acc": acc}
    return loss


# ── RUN ───────────────────────────────────────────────────────────────────────
ex01b_info = {
    "exp_id":      "EX_01b",
    "epochs":      3,
    "log_to":      "both", #"file" if you dont want wandb 
    "weights_dir": "Model/B",
    "log_dir":     "logs/B",
    "seed":        42,
}

b_model = LeNet4()
optimizer = torch.optim.Adam(b_model.parameters(), lr=config.lr)

run_experiment(
    model=b_model,
    optimizer=optimizer,
    loss_fn=ex01b_loss_fn,
    train_loader=ex01b_trainloader,
    val_loader=ex01b_valloader,
    config=ex01b_info,
    log_to=ex01b_info["log_to"],
)

[device] using cuda


[EX_01b] epoch 1/3:  16%|█▌        | 359/2265 [00:46<04:04,  7.78it/s, loss=0.2882]

In [55]:
if b_model is None:
    b_model = LeNet4().to(device)
    b_model.load_state_dict(torch.load('Model/B/EX_01b_final.pt', map_location=device))
b_model.eval()

#we can redefine this to be the 4-d version
def m_classify(image, correct):
    if image.shape[1] != 3:
        raise Exception(f"expected a 3 dim image, got a image of shape {image.shape}")
    # with torch.no_grad():
    logits = b_model(image)
    preds = torch.softmax(logits, dim=1)
    return preds[torch.arange(preds.shape[0]), correct]

So now we can rerun the above with no issues. 

# Variation C - Simple Learned Baseline

Now we can train a simple standard model to learn the mask, which is hopefully an improvement on the random one. 

In [ ]:


class baselineLearnedMask(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 3, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        mask  = self.net(x)
        return (x * mask).clamp(0, 1)

def box_blur(x, kernel_size=9):
    channels = x.shape[1]
    kernel = torch.ones(channels, 1, kernel_size, kernel_size, 
                        device=x.device) / (kernel_size ** 2)
    return F.conv2d(x, kernel, padding=kernel_size//2, groups=channels)



# ── RUN ───────────────────────────────────────────────────────────────────────
ex01c_info = {
    "exp_id":      "EX_01c",
    "epochs":      3,
    "log_to":      "both",
    "weights_dir": "Model/C",
    "log_dir":     "logs/C",
    "seed":        42,
}

model_1c = baselineLearnedMask()
optimizer = torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=1e-3)

run_experiment(
    model=model_1c,
    optimizer=optimizer,
    loss_fn=ex_01a_loss_fn,
    train_loader=ex01_trainloader,
    val_loader=ex01_trainloader,
    config=ex01c_info,
    log_to=ex01c_info["log_to"],
)

[device] using cpu


[EX_01c] epoch 2/3:   9%|▉         | 3327/37500 [00:56<09:29, 60.03it/s, loss=0.3533] 

# Evaluation

First, we compare the results of both models. 

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

def run_eval(model, loader, device, variation):
    model.eval()
    all_preds, all_labels, all_confs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)
            preds  = probs.argmax(1)
            confs  = probs.max(1).values
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())
    
    print(classification_report(all_labels, all_preds))
    df = pd.DataFrame({"label": all_labels, "pred": all_preds, "conf": all_confs})
    df.to_csv(f"/Model/{variation}/EX_01{variation}_preds.csv", index=False)
    return df

# run_eval(model_1c, ex01_trainloader, device, "C")
# run_eval(model_1a, ex01_trainloader, device, "A")
df = pd.DataFrame(np.random.randn(5, 3), columns=list('ABC'))
df.to_csv(f"/Model/C/EX_01C_preds.csv", index=False)

We might also want to see the images themselves, just to ensure nothing weird is happening

In [ ]:
import matplotlib.pyplot as plt

def show_examples(model, dataset, device, n=4, variation):
    fig, axes = plt.subplots(n, 3, figsize=(8, n*3))
    for i in range(n):
        comp, label = dataset[i]
        comp_in = comp.unsqueeze(0).to(device)
        
        with torch.no_grad():
            masked = model(comp_in)
            blurred = box_blur(masked)
        
        axes[i,0].imshow(comp.permute(1,2,0))
        axes[i,0].set_title(f"input (label={label})")
        axes[i,1].imshow(masked.squeeze(0).cpu().permute(1,2,0))
        axes[i,1].set_title("masked")
        axes[i,2].imshow(blurred.squeeze(0).cpu().permute(1,2,0))
        axes[i,2].set_title("blurred")
        for ax in axes[i]: ax.axis("off")
    
    plt.tight_layout()
    plt.savefig(f"/Model/{variation}/EX_01{variation}_samples.png")
    plt.show()